# Lab 1: Document Loading and Processing with LangChain (Free Version)

## Introduction

Welcome to the first lab in our series on Retrieval-Augmented Generation (RAG) with LangChain! This version uses completely free, open-source alternatives that don't require any API keys.

### Learning Objectives
- Load documents from PDFs, webpages, and YouTube URLs
- Split text into manageable chunks
- Extract and preserve metadata
- Use only free, open-source tools

## 1. Environment Setup

First, let's install the required packages. Run the cell below to install all dependencies:

In [18]:
# Install required packages
!pip install langchain langchain_community sentence-transformers pypdf beautifulsoup4 pytube chromadb requests

In [19]:
# Import necessary libraries
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader, YoutubeLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import requests
import os

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## 2. Loading PDF Documents

Let's start by loading a PDF document. We'll download a sample PDF from the internet and then load it using LangChain's PyPDFLoader.

In [20]:
# Download a sample PDF (the famous "Attention is All You Need" paper)
pdf_url = "https://arxiv.org/pdf/1706.03762.pdf"
pdf_filename = "attention_is_all_you_need.pdf"

# Download the PDF
response = requests.get(pdf_url)
with open(pdf_filename, "wb") as f:
    f.write(response.content)

print(f"✅ Downloaded {pdf_filename}")

✅ Downloaded attention_is_all_you_need.pdf


In [21]:
# Load the PDF using PyPDFLoader
loader = PyPDFLoader(pdf_filename)
pages = loader.load_and_split()

print(f"📄 Loaded {len(pages)} pages from the PDF")
print(f"\n📝 First page preview:")
print(pages[0].page_content[:500] + "...")
print(f"\n📋 Metadata: {pages[0].metadata}")

📄 Loaded 16 pages from the PDF

📝 First page preview:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz ...

📋 Metadata: {'source': 'attention_is_all_you_need.pdf', 'page': 0}


## 3. Loading Webpages

Now let's load content from a webpage. We'll use Wikipedia as an example since it has rich, structured content.

In [22]:
# Load a webpage
web_url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
web_loader = WebBaseLoader(web_url)
web_docs = web_loader.load()

print(f"🌐 Loaded {len(web_docs)} documents from the webpage")
print(f"\n📝 Content preview:")
print(web_docs[0].page_content[:500] + "...")
print(f"\n📋 Metadata: {web_docs[0].metadata}")

🌐 Loaded 1 documents from the webpage

📝 Content preview:




Artificial intelligence - Wikipedia


























Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us





		Contribute
	


HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages



















Search











Search






















Appearance
















Donate

Create account

Log in








Personal tools





Donate Create account Log in...

📋 Metadata: {'source': 'https://en.wikipedia.org/wiki/Artificial_intelligence', 'title': 'Artificial intelligence - Wikipedia', 'language': 'en'}


## 4. Loading YouTube Transcripts

Let's load a transcript from a YouTube video. We'll use a popular educational video about neural networks.

In [23]:
%pip install --upgrade --quiet  youtube-transcript-api

Note: you may need to restart the kernel to use updated packages.


# Load YouTube transcript
# Using 3Blue1Brown's "But what is a neural network?" video
youtube_url = "https://www.youtube.com/watch?v=aircAruvnKk"

youtube_loader = YoutubeLoader.from_youtube_url(
    youtube_url, 
    add_video_info=True
)
youtube_docs = youtube_loader.load()

print(f"🎥 Loaded transcript from YouTube video")
print(f"📺 Video title: {youtube_docs[0].metadata.get('title', 'Unknown')}")
print(f"👤 Author: {youtube_docs[0].metadata.get('author', 'Unknown')}")
print(f"\n📝 Transcript preview:")
print(youtube_docs[0].page_content[:500] + "...")

## 5. Text Splitting

Now that we have loaded documents from different sources, we need to split them into smaller chunks. This is crucial for RAG systems because:
1. LLMs have limited context windows
2. Smaller chunks provide more precise retrieval
3. Better embedding quality for focused content

In [24]:
# Create a text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # Maximum chunk size
    chunk_overlap=200,  # Overlap between chunks to maintain context
    length_function=len,  # Function to measure chunk length
    separators=["\n\n", "\n", " ", ""]  # Preferred split points
)

print("✅ Text splitter created")

✅ Text splitter created


In [25]:
# Split the PDF pages
pdf_splits = text_splitter.split_documents(pages)

print(f"📄 PDF split into {len(pdf_splits)} chunks")
print(f"\n📝 First chunk preview:")
print(pdf_splits[0].page_content)
print(f"\n📏 Chunk length: {len(pdf_splits[0].page_content)} characters")

📄 PDF split into 52 chunks

📝 First chunk preview:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispen

In [26]:
# Split the webpage content
web_splits = text_splitter.split_documents(web_docs)

print(f"🌐 Webpage split into {len(web_splits)} chunks")
print(f"\n📝 First chunk preview:")
print(web_splits[0].page_content[:300] + "...")

🌐 Webpage split into 330 chunks

📝 First chunk preview:
Artificial intelligence - Wikipedia


























Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us





		Contribute
	


HelpLearn to editCommunity portalRecent changesUpload fi...


# Split the YouTube transcript
youtube_splits = text_splitter.split_documents(youtube_docs)

print(f"🎥 YouTube transcript split into {len(youtube_splits)} chunks")
print(f"\n📝 First chunk preview:")
print(youtube_splits[0].page_content[:300] + "...")

## 6. Combining All Documents

Let's combine all our processed documents into a single collection for further processing.

In [28]:
# Combine all splits
all_splits = pdf_splits + web_splits #+ youtube_splits

print(f"📚 Total documents: {len(all_splits)}")
print(f"   - PDF chunks: {len(pdf_splits)}")
print(f"   - Web chunks: {len(web_splits)}")
#print(f"   - YouTube chunks: {len(youtube_splits)}")

# Show source distribution
sources = {}
for doc in all_splits:
    source = doc.metadata.get('source', 'unknown')
    sources[source] = sources.get(source, 0) + 1

print(f"\n📊 Source distribution:")
for source, count in sources.items():
    print(f"   - {source}: {count} chunks")

📚 Total documents: 382
   - PDF chunks: 52
   - Web chunks: 330

📊 Source distribution:
   - attention_is_all_you_need.pdf: 52 chunks
   - https://en.wikipedia.org/wiki/Artificial_intelligence: 330 chunks


## 7. Exercises

Now it's time to practice! Complete the following exercises to reinforce your learning.

### Exercise 1: Load a Different PDF

Find another PDF online (perhaps a research paper or documentation) and load it using PyPDFLoader.

In [29]:
# Exercise 1: Your code here
# TODO: 
# 1. Find a PDF URL
# 2. Download it
# 3. Load it with PyPDFLoader
# 4. Print some statistics about it

# Your code here:


### Exercise 2: Load Multiple Webpages

Load content from 3 different webpages related to machine learning or AI.

In [30]:
# Exercise 2: Your code here
# TODO:
# 1. Define a list of 3 URLs
# 2. Load each webpage
# 3. Combine all documents
# 4. Print statistics

# Your code here:


### Exercise 3: Custom Text Splitter

Create a custom text splitter with different parameters and compare the results.

In [31]:
# Exercise 3: Your code here
# TODO:
# 1. Create a text splitter with chunk_size=500 and chunk_overlap=50
# 2. Split one of your documents
# 3. Compare with the original splitter results
# 4. Analyze the differences

# Your code here:


### Exercise 4: Metadata Enhancement

Add custom metadata to your documents (e.g., topic, difficulty level, source type).

In [32]:
# Exercise 4: Your code here
# TODO:
# 1. Take some of your loaded documents
# 2. Add custom metadata fields
# 3. Create new Document objects with enhanced metadata
# 4. Print the enhanced metadata

# Your code here:


## Solutions

Here are the solutions to the exercises. Try to complete them yourself first!

In [33]:
# Solution 1: Load a different PDF
# Using BERT paper as an example
bert_url = "https://arxiv.org/pdf/1810.04805.pdf"
bert_filename = "bert_paper.pdf"

response = requests.get(bert_url)
with open(bert_filename, "wb") as f:
    f.write(response.content)

bert_loader = PyPDFLoader(bert_filename)
bert_pages = bert_loader.load_and_split()

print(f"📄 BERT paper: {len(bert_pages)} pages")
print(f"📝 First page preview: {bert_pages[0].page_content[:200]}...")

📄 BERT paper: 24 pages
📝 First page preview: BERT: Pre-training of Deep Bidirectional Transformers for
Language Understanding
Jacob Devlin Ming-Wei Chang Kenton Lee Kristina Toutanova
Google AI Language
{jacobdevlin,mingweichang,kentonl,kristout...


In [34]:
# Solution 2: Load multiple webpages
urls = [
    "https://en.wikipedia.org/wiki/Machine_learning",
    "https://en.wikipedia.org/wiki/Deep_learning",
    "https://en.wikipedia.org/wiki/Natural_language_processing"
]

all_web_docs = []
for url in urls:
    loader = WebBaseLoader(url)
    docs = loader.load()
    all_web_docs.extend(docs)
    print(f"✅ Loaded: {url}")

print(f"\n📚 Total web documents: {len(all_web_docs)}")

✅ Loaded: https://en.wikipedia.org/wiki/Machine_learning
✅ Loaded: https://en.wikipedia.org/wiki/Deep_learning
✅ Loaded: https://en.wikipedia.org/wiki/Natural_language_processing

📚 Total web documents: 3


In [35]:
# Solution 3: Custom text splitter
custom_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# Compare with original splitter
original_splits = text_splitter.split_documents(pages[:1])  # First page only
custom_splits = custom_splitter.split_documents(pages[:1])

print(f"📊 Comparison for first page:")
print(f"   Original splitter: {len(original_splits)} chunks")
print(f"   Custom splitter: {len(custom_splits)} chunks")
print(f"\n📏 Average chunk sizes:")
print(f"   Original: {sum(len(doc.page_content) for doc in original_splits) / len(original_splits):.0f} chars")
print(f"   Custom: {sum(len(doc.page_content) for doc in custom_splits) / len(custom_splits):.0f} chars")

📊 Comparison for first page:
   Original splitter: 4 chunks
   Custom splitter: 7 chunks

📏 Average chunk sizes:
   Original: 831 chars
   Custom: 420 chars


In [36]:
# Solution 4: Metadata enhancement
from langchain.schema import Document

# Enhance PDF documents
enhanced_docs = []
for doc in pdf_splits[:3]:  # First 3 chunks
    enhanced_metadata = doc.metadata.copy()
    enhanced_metadata.update({
        'topic': 'transformer_architecture',
        'difficulty': 'advanced',
        'source_type': 'academic_paper',
        'year': 2017
    })
    
    enhanced_doc = Document(
        page_content=doc.page_content,
        metadata=enhanced_metadata
    )
    enhanced_docs.append(enhanced_doc)

print("📋 Enhanced metadata example:")
for key, value in enhanced_docs[0].metadata.items():
    print(f"   {key}: {value}")

📋 Enhanced metadata example:
   source: attention_is_all_you_need.pdf
   page: 0
   topic: transformer_architecture
   difficulty: advanced
   source_type: academic_paper
   year: 2017


## Summary

Congratulations! You've completed Lab 1. Here's what you've learned:

✅ **Document Loading**: How to load documents from PDFs, webpages, and YouTube videos using LangChain loaders

✅ **Text Splitting**: How to split large documents into manageable chunks using RecursiveCharacterTextSplitter

✅ **Metadata Management**: How to work with and enhance document metadata

✅ **Free Tools**: How to build a document processing pipeline using only free, open-source tools

### Next Steps

In Lab 2, we'll take these processed documents and:
- Generate embeddings using free Hugging Face models
- Store them in a vector database (ChromaDB)
- Perform similarity search to retrieve relevant documents

### Key Takeaways

1. **Different sources require different loaders** - PDFs, web content, and video transcripts each have specialized loaders
2. **Text splitting is crucial** - Proper chunking affects retrieval quality and LLM performance
3. **Metadata is valuable** - Rich metadata enables better filtering and organization
4. **Free tools are powerful** - You can build sophisticated document processing pipelines without paid APIs

Great job! 🎉